In [3]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.load_criteo import load_counts, load_sample
from src.simulate import simulate_experiment
from src.srm import srm_check, srm_by_segment

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

# 02 Loading and Sanity Checks

Nothing in this notebook looks at an outcome metric. That is deliberate. The
sanity checks have to pass before we are allowed to read the result, because
knowing the result contaminates your ability to investigate a problem impartially.
Once you have seen a positive lift, you will look for reasons the SRM does not
matter. Check first.

In [4]:
counts = load_counts()

print(f"Rows in the full file : {counts['n_total']:,}")
print(f"Control               : {counts['n_control']:>12,}  ({counts['control_share']:.2%})")
print(f"Treatment             : {counts['n_treatment']:>12,}  ({counts['treatment_share']:.2%})")
print()
print("These are exact counts over all fourteen million rows, accumulated while")
print("streaming the file in chunks. We never held the whole frame in memory.")
print("The two proportion test consumes four integers, so it does not need us to.")

Rows in the full file : 13,979,592.0
Control               :  2,096,937.0  (15.00%)
Treatment             : 11,882,655.0  (85.00%)

These are exact counts over all fourteen million rows, accumulated while
streaming the file in chunks. We never held the whole frame in memory.
The two proportion test consumes four integers, so it does not need us to.


In [5]:
criteo_srm = srm_check(
    observed_counts={
        "control": int(counts["n_control"]),
        "treatment": int(counts["n_treatment"]),
    },
    expected_ratios={"control": 0.15, "treatment": 0.85},
)

for k, v in criteo_srm.items():
    print(f"{k:16s} {v}")

arms             ['control', 'treatment']
observed         [2096937, 11882655]
expected         [2096938.8, 11882653.2]
observed_ratio   [0.15, 0.85]
expected_ratio   [0.15, 0.85]
chi2             1.8177758481483255e-06
p_value          0.9989242531220275
alpha            0.001
passed           True
verdict          No sample ratio mismatch. The randomisation behaved as designed. Safe to proceed.


## The trap this dataset sets

Criteo runs an 85 / 15 split on purpose. Withholding advertising from users costs
money, so the control arm is kept as small as the required power will allow.

A naive SRM implementation tests whether the split is balanced. Run that here and
it fires immediately with an astronomically small p value, and you would conclude
the experiment is broken when it is working exactly as designed.

The test is always against the intended allocation, never against balance. The
cell below shows what the wrong version would have told us.

wrong = srm_check(
    observed_counts={
        "control": int(counts["n_control"]),
        "treatment": int(counts["n_treatment"]),
    },
    expected_ratios={"control": 0.5, "treatment": 0.5},
)

print("Testing against a 50 / 50 assumption:")
print(f"  chi2    {wrong['chi2']:,.0f}")
print(f"  p value {wrong['p_value']:.2e}")
print(f"  passed  {wrong['passed']}")
print()
print("A confident, precise, completely wrong answer. This is what happens when")
print("a check is copied from a tutorial without reading the experiment design.")

In [6]:
df_sim, ground_truth = simulate_experiment(n_users=80_000, seed=42)
df_sim.to_parquet("../data/processed/simulated_onboarding.parquet", index=False)

sim_counts = df_sim["group"].value_counts().to_dict()
sim_srm = srm_check(sim_counts, {"control": 0.5, "treatment": 0.5})

print(f"Simulated users: {len(df_sim):,}")
print(f"Split: {sim_srm['observed_ratio']}")
print(f"p value: {sim_srm['p_value']:.4f}")
print(sim_srm["verdict"])

Simulated users: 80,000
Split: [0.50321, 0.49679]
p value: 0.0692
No sample ratio mismatch. The randomisation behaved as designed. Safe to proceed.


In [7]:
df_broken, _ = simulate_experiment(n_users=80_000, seed=7, inject_srm=True)
broken_counts = df_broken["group"].value_counts().to_dict()
broken = srm_check(broken_counts, {"control": 0.5, "treatment": 0.5})

observed_pct = {k: f"{v / len(df_broken):.1%}" for k, v in broken_counts.items()}

print("Injected a broken assignment. The observed split:")
print(f"  {observed_pct}")
print()
print("Nobody eyeballing that on a dashboard would stop the experiment.")
print()
print(f"The chi square test: p = {broken['p_value']:.2e}")
print(broken["verdict"])

Injected a broken assignment. The observed split:
  {'treatment': '53.0%', 'control': '47.0%'}

Nobody eyeballing that on a dashboard would stop the experiment.

The chi square test: p = 6.46e-64
SRM DETECTED. Do not interpret any metric. Investigate assignment and logging before reading a single result.


## Why the SRM check earns its place

The broken split above is roughly 48.5 / 51.5. That is well inside the range a
human would call "close enough". At eighty thousand users it is essentially
impossible by chance, and the chi square test says so immediately.

In production this pattern usually means one of three things.

Bot filtering removing traffic asymmetrically, because bots behave differently in
the two arms and the filter catches them at different rates.

A redirect or feature flag failing on one platform, so some treatment users
silently fall back to control.

An exposure event that only fires after the page renders, so treatment users on
slow connections drop out before they are ever logged, which selectively removes
exactly the users least likely to convert.

All three bias the sample. None of them are visible in the conversion numbers.
The only place they show up is the assignment counts.

segment_results = srm_by_segment(
    df_sim, group_col="group", segment_col="device",
    expected_ratios={"control": 0.5, "treatment": 0.5},
)

rows = []
for device, r in segment_results.items():
    ci = r["arms"].index("control")
    ti = r["arms"].index("treatment")
    rows.append({
        "device": device,
        "control": r["observed"][ci],
        "treatment": r["observed"][ti],
        "p_value": round(r["p_value"], 4),
        "passed": r["passed"],
    })

print("Segment level SRM. Run this even when the total passes, because two")
print("opposing segment failures can cancel into a clean overall number.")
pd.DataFrame(rows)

In [8]:
sample = load_sample()

features = [f"f{i}" for i in range(12)]
c = sample[sample["treatment"] == 0]
t = sample[sample["treatment"] == 1]

balance = pd.DataFrame({
    "feature": features,
    "mean_control": [c[f].mean() for f in features],
    "mean_treatment": [t[f].mean() for f in features],
})
balance["std_diff"] = (
    (balance["mean_treatment"] - balance["mean_control"])
    / sample[features].std().values
)

print("Standardised mean difference on the covariates.")
print("Randomisation should make these near zero. Anything above about 0.1 is")
print("worth a second look, because it suggests the arms are not comparable")
print("on something we can see, which raises the question of what they differ")
print("on that we cannot.")
print()
balance.round(4)

Standardised mean difference on the covariates.
Randomisation should make these near zero. Anything above about 0.1 is
worth a second look, because it suggests the arms are not comparable
on something we can see, which raises the question of what they differ
on that we cannot.



,feature,mean_control,mean_treatment,std_diff
0,f0,19.67170,19.61870,-0.00990
1,f1,10.06750,10.07040,0.02750
2,f2,8.44840,8.44620,-0.00750
3,f3,4.23540,4.16970,-0.04920
4,f4,10.33570,10.33970,0.01160
5,f5,4.04010,4.02700,-0.03070
6,f6,-4.00160,-4.18360,-0.03980
7,f7,5.07920,5.10350,0.02030
8,f8,3.93470,3.93340,-0.02440
9,f9,15.86590,16.05840,0.02740
